# Guardião da Família — notebook do Colab

O Guardião avisa um familiar responsável quando um **número novo** diz ser parente de uma
pessoa idosa e pede dinheiro (o golpe do falso parente). Este notebook instala o backend,
lê as credenciais dos **Secrets do Colab** e roda o Guardião em um de dois modos:

- **demonstração** (padrão): a conta da pessoa idosa é simulada, com contatos, conversas
  antigas e números fictícios (DDD 00, que não existe no Brasil). O bot do Telegram que
  alerta a responsável continua real.
- **real**: lê a conta de teste do Telegram da pessoa idosa pelo Telethon.

Rode as células em ordem. Já funcionam a recepção das mensagens, o filtro de contato novo,
o cadastro da responsável pelo bot, a detecção do falso parente pelo Gemini e a busca do
número verdadeiro nos contatos. Os alertas, os casos e o desfecho entram nas próximas
versões.

**Segurança:** nenhuma credencial é escrita neste notebook. Elas ficam nos Secrets do Colab
(ícone de chave na barra lateral) e o notebook só mostra se cada uma está *presente* ou
*ausente*. Nunca cole um token ou chave numa célula.

## 1. Obter o código

A célula abaixo procura a pasta do projeto (com `backend/` e `frontend/`) nesta ordem:

1. **pasta existente** — a pasta atual, as pastas acima dela ou uma pasta em `/content`;
2. **ZIP enviado** — um `.zip` do projeto enviado para `/content` (painel *Arquivos*), que é extraído;
3. **`git clone`** do repositório público, só no Colab. Não precisa de token.

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

try:
    import google.colab  # noqa: F401

    EM_COLAB = True
except ImportError:
    EM_COLAB = False

REPOSITORIO = "https://github.com/fiap-poa/guardiao_gs_fiap.git"
CONTENT = Path("/content")
DESTINO_CLONE = CONTENT / "guardiao_gs_fiap"


def _eh_raiz(pasta):
    return (pasta / "backend" / "guardiao").is_dir() and (pasta / "frontend").is_dir()


def _procurar_raiz():
    candidatos = [Path.cwd(), *Path.cwd().parents]
    if CONTENT.is_dir():
        candidatos += sorted(p for p in CONTENT.glob("*") if p.is_dir())
        candidatos += sorted(p for p in CONTENT.glob("*/*") if p.is_dir())
    return next((p.resolve() for p in candidatos if _eh_raiz(p)), None)


def _extrair_zip():
    if not CONTENT.is_dir():
        return False
    for arquivo in sorted(CONTENT.glob("*.zip")):
        with zipfile.ZipFile(arquivo) as zip_:
            if any(nome.endswith("backend/guardiao/__init__.py") for nome in zip_.namelist()):
                destino = CONTENT / arquivo.stem
                zip_.extractall(destino)
                print(f"ZIP extraído: {arquivo.name} -> {destino}")
                return True
    return False


def _clonar():
    # repositório público: sem token; GIT_TERMINAL_PROMPT=0 faz o git falhar em vez de travar pedindo usuário
    ambiente = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    comando = ["git", "clone", "--depth", "1", "-q", REPOSITORIO, str(DESTINO_CLONE)]
    print("clonando o repositório...")
    try:
        resultado = subprocess.run(comando, capture_output=True, text=True, env=ambiente, timeout=120)
        codigo, erro = resultado.returncode, resultado.stderr.strip()
    except subprocess.TimeoutExpired:
        codigo, erro = None, "tempo esgotado (120 s)"
    if codigo != 0:
        raise RuntimeError(f"git clone falhou (código {codigo}): {erro}")


RAIZ = _procurar_raiz()
if RAIZ is None and _extrair_zip():
    RAIZ = _procurar_raiz()
if RAIZ is None:
    if EM_COLAB:
        _clonar()
        RAIZ = _procurar_raiz()
    else:
        print("aviso: fora do Colab o clone é pulado; abra este notebook de dentro do repositório")
if RAIZ is None:
    raise RuntimeError("não achei a pasta do projeto (com backend/ e frontend/)")

BACKEND = RAIZ / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
print(f"projeto em {RAIZ} (Colab: {'sim' if EM_COLAB else 'não'})")

## 2. Instalar as dependências

No Colab, instala `backend/requirements.txt` (Flask, Telethon, cliente HTTP da Bot API…).
Fora do Colab a instalação é pulada: use o ambiente virtual do backend
(`backend/.venv`) como kernel.

In [ ]:
import importlib.util

REQUISITOS = BACKEND / "requirements.txt"
if EM_COLAB:
    instalacao = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUISITOS)], capture_output=True, text=True
    )
    if instalacao.returncode != 0:
        print("\n".join(instalacao.stderr.strip().splitlines()[-20:]))
        raise RuntimeError(f"pip install falhou (código {instalacao.returncode})")
    print("dependências instaladas")
else:
    faltando = [m for m in ("flask", "telethon", "requests", "dotenv", "qrcode") if importlib.util.find_spec(m) is None]
    try:
        tem_genai = importlib.util.find_spec("google.genai") is not None
    except ModuleNotFoundError:  # nem o pacote `google` existe
        tem_genai = False
    if not tem_genai:
        faltando.append("google-genai")
    if faltando:
        print(f"aviso: fora do Colab a instalação é pulada; faltam {faltando}: pip install -r {REQUISITOS}")
    else:
        print("aviso: fora do Colab a instalação é pulada (as dependências já estão no kernel)")

## 3. Credenciais (Secrets do Colab)

No Colab, abra o ícone de **chave** na barra lateral, crie os Secrets abaixo e ative
*Acesso ao notebook* em cada um:

| Secret | Para quê | Modo |
|---|---|---|
| `TELEGRAM_BOT_TOKEN` | bot do Guardião (alerta a responsável) | demonstração e real |
| `GEMINI_API_KEY` | detecção do falso parente nas mensagens de contato novo | demonstração e real |
| `GEMINI_MODELO` | opcional: outro modelo do Gemini (padrão `gemini-3.5-flash-lite`) | demonstração e real |
| `TELEGRAM_API_ID`, `TELEGRAM_API_HASH` | ler a conta de teste pelo Telethon | só real |

A célula copia os Secrets para variáveis de ambiente do processo e mostra só
**presente/ausente**. Fora do Colab, as credenciais vêm de `backend/.env` (fora do git).
Sem `TELEGRAM_BOT_TOKEN`, o modo demonstração avisa e segue sem bot: os cenários rodam,
mas nenhum alerta chega à responsável. Sem `GEMINI_API_KEY`, a detecção fica desligada:
o filtro funciona, mas nada é analisado.

In [ ]:
from guardiao.config import carregar_env, carregar_segredos_colab

SEGREDOS = ["TELEGRAM_BOT_TOKEN", "GEMINI_API_KEY", "GEMINI_MODELO", "TELEGRAM_API_ID", "TELEGRAM_API_HASH"]
do_colab = carregar_segredos_colab(SEGREDOS)
if not EM_COLAB:
    print("aviso: fora do Colab não há Secrets; lendo backend/.env\n")
    carregar_env()
for nome in SEGREDOS:
    presente = do_colab[nome] or bool(os.environ.get(nome, "").strip())
    print(f"{nome:20} {'presente' if presente else 'ausente'}")
if not os.environ.get("TELEGRAM_BOT_TOKEN", "").strip():
    print("\naviso: sem TELEGRAM_BOT_TOKEN o modo demonstração segue sem bot (nenhum alerta sai)")
if not os.environ.get("GEMINI_API_KEY", "").strip():
    print("\naviso: sem GEMINI_API_KEY a detecção pelo Gemini fica desligada")

## 4. Escolher o modo

- `"demonstracao"`: canal simulado. É o modo da entrega e não precisa de conta do Telegram.
- `"real"`: Telethon numa conta **de teste** do Telegram (nunca a conta de uma pessoa real).
  Precisa de `TELEGRAM_API_ID`, `TELEGRAM_API_HASH` e de uma sessão já logada por QR;
  é opcional.

In [ ]:
MODO = "demonstracao"  # ou "real"

from guardiao.canal.fabrica import MODOS

if MODO not in MODOS:
    raise ValueError(f"MODO deve ser um de {MODOS}")
print(f"modo: {MODO}")

## 5. Conectar o Guardião

O canal é a única parte que fala com o mensageiro; o **Guardião** recebe tudo o que ele
entrega. Ao conectar, ele:

1. registra como **conhecidos** os contatos que já tinham conversa privada com a pessoa
   idosa (e o próprio bot);
2. liga a recepção das mensagens;
3. manda o `/start` automático pela conta da pessoa idosa, para o bot poder avisá-la
   depois (o bot não responde a ela).

Daí em diante, só as mensagens de **contato novo** seguem para a análise (guardadas em
`ANALISADAS`). Grupos e conhecidos são descartados sem ler o texto. Com `GEMINI_API_KEY`,
cada mensagem de contato novo vai ao **Gemini** junto com as anteriores do mesmo número
nos últimos 30 minutos (só os textos: nenhum contato, telefone ou nome da agenda). Se a
pessoa diz ser parente **e** pede dinheiro, o Guardião busca nos contatos o número
verdadeiro desse parente e guarda a suspeita em `SUSPEITAS` (o alerta à responsável vem
na próxima versão).

**Privacidade:** os textos analisados vão para o Google (Gemini API). No nível gratuito, os
termos da Gemini API permitem que o Google use esse conteúdo para melhorar os produtos.
No modo demonstração as mensagens são fictícias; no modo real, use só contas de teste da
equipe, nunca a conta de uma pessoa real. O relógio do Guardião (`RELOGIO`) pode ser adiantado para mostrar a regra dos
7 dias. Rodar a célula de novo fecha o Guardião anterior antes de criar outro.

In [ ]:
from guardiao.analise import Analisador
from guardiao.app import definir_guardiao
from guardiao.canal.fabrica import criar_canal
from guardiao.gemini import criar_detector
from guardiao.recepcao import Guardiao, RelogioAjustavel

if globals().get("guardiao") is not None:
    guardiao.fechar()
elif globals().get("canal") is not None:
    canal.fechar()
canal = guardiao = None  # se a criação falhar, não sobra referência ao canal fechado

RELOGIO = RelogioAjustavel()  # a demonstração dos 7 dias o adianta
ANALISADAS = []  # mensagens de contato novo que seguem para a análise
RESPOSTAS = []  # toques da responsável nos botões dos alertas (próximas versões)
SUSPEITAS = []  # contatos novos que disseram ser parente e pediram dinheiro

detector = criar_detector()  # None sem GEMINI_API_KEY
canal = criar_canal(MODO)
analisador = None
if detector is not None:
    analisador = Analisador(detector, canal.listar_contatos, ao_suspeitar=SUSPEITAS.append, relogio=RELOGIO)
guardiao = Guardiao(
    canal, relogio=RELOGIO, ao_analisar=ANALISADAS.append, ao_responder=RESPOSTAS.append, analisador=analisador
)
try:
    estado = guardiao.conectar()
except Exception as erro:
    guardiao = None
    print(f"aviso: o Guardião não conectou ({type(erro).__name__})")
else:
    definir_guardiao(guardiao)  # a tela (seção 8) passa a mostrar este Guardião
    print(f"conectado: {estado['conhecidos']} contatos conhecidos no pareamento")
    print(f"pessoa idosa habilitada no bot: {'sim' if estado['idosa_habilitada'] else 'aguardando o /start'}")
    if detector is None:
        print("detecção pelo Gemini: desligada (sem GEMINI_API_KEY)")
    else:
        print(f"detecção pelo Gemini: ligada (modelo {detector.cliente.modelo})")
print(canal)

## 6. Cenários do modo demonstração

A pessoa idosa simulada é a **Dona Maria**. Os contatos e conversas antigas são fictícios,
com DDD 00. Os cinco cenários:

| Cenário | O que acontece | Esperado |
|---|---|---|
| `golpe_classico` | número novo diz ser o neto Pedro e pede Pix | alerta |
| `parente_trocou_numero` | o filho Lucas trocou de número de verdade e pede transferência | alerta, e a responsável responde "Era ele mesmo" |
| `contato_antigo_pede_dinheiro` | o vizinho Carlos, que já conversava com ela, pede dinheiro | ignorado |
| `mensagem_em_grupo` | alguém no grupo Família pede Pix | ignorado (o Guardião não lê grupos) |
| `contato_novo_sem_dinheiro` | número novo diz ser a neta Júlia e só manda saudades | sem alerta |

In [ ]:
from guardiao.canal.cenarios import CENARIOS

for cenario in CENARIOS:
    print(f"{cenario.id:30} esperado={cenario.esperado:22} {cenario.descricao}")

if MODO == "demonstracao":
    print("\nContatos salvos no celular simulado:")
    for contato in canal.listar_contatos():
        print(f"  {contato.nome:16} {contato.telefone}")
    print("\nConversas que já existiam:")
    for conversa in canal.conversas_existentes():
        print(f"  {conversa.nome:16} ({conversa.tipo})")

### Injetar os cinco cenários

Cada mensagem do cenário chega ao canal simulado como se tivesse chegado ao Telegram da
Dona Maria, e o Guardião decide: **análise** (contato novo; o texto aparece porque é
fictício) ou **descarte** (grupo ou conhecido; o texto não é lido nem mostrado). Só as
mensagens dos três números novos seguem para a análise. Com o Gemini ligado, cada uma
delas é analisada na hora (pode levar alguns segundos) e a linha `Gemini:` mostra o
resultado; o detalhe vem na célula seguinte.

In [ ]:
from guardiao.canal.simulado import descrever_recebido

MOTIVOS_DESCARTE = {
    "grupo": "grupo",
    "conhecido": "contato conhecido",
    "observacao_encerrada": "7 dias sem alerta: virou conhecido",
    "sem_remetente": "sem remetente",
}


SITUACOES = {
    "suspeita": "SUSPEITA: diz ser parente e pede dinheiro",
    "sem_suspeita": "sem suspeita",
    "ja_suspeito": "já marcado como suspeito (não reanalisa)",
    "falha": "falha ao consultar o Gemini",
    "sem_texto": "mensagem sem texto",
}


def injetar_mostrando(injetar, *args, **kwargs):
    """Injeta e mostra a decisão do Guardião, sem mostrar o texto do que foi descartado."""
    ja_analisadas = len(ANALISADAS)
    analisador = guardiao.analisador
    antes_resultados = analisador.ultimos() if analisador is not None else []
    antes = guardiao.filtro.contagens()["por_motivo"]
    injetar(*args, **kwargs)
    for mensagem in ANALISADAS[ja_analisadas:]:
        print("   análise:", descrever_recebido(mensagem))
    if analisador is not None:
        novos = [r for r in analisador.ultimos() if not any(r is a for a in antes_resultados)]
        for resultado in novos:
            print(f"   Gemini: {SITUACOES[resultado.situacao]}")
    depois = guardiao.filtro.contagens()["por_motivo"]
    for motivo, rotulo in MOTIVOS_DESCARTE.items():
        descartadas = depois.get(motivo, 0) - antes.get(motivo, 0)
        if descartadas:
            print(f"   descarte ({rotulo}): {descartadas} mensagem(ns), texto não lido")


if MODO != "demonstracao":
    print("aviso: a injeção de cenários só existe no modo demonstração; pulando")
elif guardiao is None:
    print("aviso: o Guardião não está conectado; rode a célula da seção 5")
else:
    for cenario in CENARIOS:
        print(f"== {cenario.id} (esperado: {cenario.esperado})")
        injetar_mostrando(canal.injetar, cenario.id)
    print(f"\nseguiram para a análise: {len(ANALISADAS)} mensagens de {len({m.remetente_id for m in ANALISADAS})} números novos")

### Detecção e busca do parente

Para cada mensagem analisada, o que o Gemini extraiu da conversa (quem a pessoa diz ser,
o parentesco, se pede dinheiro, como e por quê) e, quando há suspeita, o número que a
Dona Maria tem salvo para esse nome. A suspeita é decidida pelo Guardião, não pelo
modelo: só quando há alegação de parentesco **e** pedido de dinheiro. Esperado: o golpe
clássico (Pedro) e o filho que trocou de número (Lucas) marcam suspeita, com o número
verdadeiro de cada um; a neta Júlia, que só manda saudades, não. Contato antigo e grupo
nem chegam ao Gemini.

Sem correspondência nos contatos, o alerta segue sem número sugerido. Depois da suspeita,
a conversa daquele número sai da memória e ele não é reanalisado.


In [ ]:
SIM_NAO = {True: "sim", False: "não"}
BUSCAS = {
    "encontrado": "número salvo nos contatos",
    "sem_correspondencia": "não achei esse nome nos contatos: o alerta segue sem número",
    "ambiguo": "mais de um contato com esse nome: o alerta segue sem número",
    "sem_nome": "a pessoa não disse o nome: o alerta segue sem número",
    "sem_telefone": "o contato não tem telefone salvo: o alerta segue sem número",
    "sem_contatos": "não consegui ler os contatos: o alerta segue sem número",
}

if guardiao is None:
    print("aviso: o Guardião não está conectado; rode a célula da seção 5")
elif guardiao.analisador is None:
    print("aviso: a detecção está desligada (sem GEMINI_API_KEY): crie o Secret e rode as seções 3 e 5 de novo")
elif not guardiao.analisador.ultimos():
    print("nada analisado ainda: rode a injeção dos cenários acima")
else:
    for resultado in guardiao.analisador.ultimos():  # cópia: no modo real, chegam resultados de outra thread
        ficticio = str(resultado.remetente_id).startswith("sim:")  # conta real: sem textos
        print(f"== {resultado.remetente_id if ficticio else 'contato novo'}: {SITUACOES[resultado.situacao]}")
        analise = resultado.analise
        if analise is not None:
            print(f"   mensagens na janela: {resultado.mensagens_analisadas}")
            quem = f" ({analise.nome_alegado or 'sem nome'}, {analise.parentesco or 'parentesco não dito'})"
            print(f"   diz ser parente: {SIM_NAO[analise.alega_parentesco]}{quem if ficticio and analise.alega_parentesco else ''}")
            como = f" ({analise.forma_pagamento}; motivo: {analise.motivo or 'não disse'})"
            print(f"   pede dinheiro: {SIM_NAO[analise.pede_dinheiro]}{como if ficticio and analise.pede_dinheiro else ''}")
        if resultado.erro:
            print(f"   erro: {resultado.erro} (a mensagem fica na janela para a próxima análise)")
        if resultado.suspeita:
            busca = resultado.busca
            if ficticio:
                print(f'   trecho: "{resultado.trecho}"')
            if busca.numero and ficticio:
                print(f"   {BUSCAS[busca.motivo]}: {busca.contato.nome} {busca.numero}")
            else:
                print(f"   {BUSCAS[busca.motivo]}")
    print(f"\nsuspeitas: {len(SUSPEITAS)}")


### Conjunto de teste do CAP-3 no Gemini real (opcional)

Oito conversas fictícias de contato novo, com a suspeita esperada: os três cenários de
número novo e cinco casos extras com **só uma** das duas condições (parente que não pede
dinheiro, banco ou amiga que pedem) ou com outras formas de pagamento (boleto, depósito).
Faz 8 chamadas ao Gemini e não passa pelo Guardião. Para rodar, mude
`RODAR_CONJUNTO_CAP3` para `True`.


In [ ]:
RODAR_CONJUNTO_CAP3 = False  # True para rodar (8 chamadas ao Gemini)

from guardiao.conjunto_cap3 import CONJUNTO_CAP3

if not RODAR_CONJUNTO_CAP3:
    print("pulando: mude RODAR_CONJUNTO_CAP3 para True para rodar o conjunto no Gemini")
elif detector is None:
    print("aviso: sem GEMINI_API_KEY não há Gemini; pulando")
else:
    acertos = 0
    for conversa in CONJUNTO_CAP3:
        esperado = "suspeita" if conversa.suspeita else "sem suspeita"
        try:
            analise = detector.analisar(list(conversa.mensagens))
        except Exception as erro:
            print(f"{conversa.id:30} esperado={esperado:13} falha ({type(erro).__name__})")
            continue
        obtido = "suspeita" if analise.suspeita else "sem suspeita"
        acertos += obtido == esperado
        print(f"{conversa.id:30} esperado={esperado:13} gemini={obtido:13} {'ok' if obtido == esperado else 'ERRO'}")
    print(f"\nacertos: {acertos} de {len(CONJUNTO_CAP3)}")


### Contato novo: observado por 7 dias

Estar salvo nos contatos não torna ninguém conhecido: a neta Júlia está salva
(`sim:julia`), mas nunca conversou com a Dona Maria, então a mensagem dela vai para a
análise. Um contato novo fica **observado** por 7 dias a partir da primeira mensagem. A
célula adianta o relógio do Guardião em 7 dias:

- o número novo que diz ser a Júlia (`sim:novo-3`, do cenário sem pedido de dinheiro, sem
  alerta) vira **conhecido**, e a mensagem dele é descartada;
- o número do golpe clássico (`sim:novo-1`) teve alerta (aqui registrado à mão; nas
  próximas versões, o alerta faz isso) e continua **observado**.

Rode depois da injeção dos cenários, com o Guardião recém-conectado (seção 5). No fim, o
relógio volta ao normal.

In [ ]:
from guardiao.canal.cenarios import obter_cenario

if MODO != "demonstracao":
    print("aviso: a demonstração dos 7 dias usa o canal simulado; pulando")
elif guardiao is None:
    print("aviso: o Guardião não está conectado; rode a célula da seção 5")
elif guardiao.filtro.situacao("sim:novo-3") != "observado":
    print("aviso: o número novo que diz ser a Júlia (sim:novo-3) não está observado;")
    print("rode a seção 5 (Guardião recém-conectado) e a injeção dos cenários antes desta célula; pulando")
else:
    print("== sim:julia (salva nos contatos, sem conversa anterior)")
    injetar_mostrando(canal.injetar_mensagem, "sim:julia", "Oi vó, a senhora está bem?", de="Júlia Neta")

    guardiao.filtro.registrar_alerta("sim:novo-1")  # o golpe clássico teve alerta
    RELOGIO.adiantar(7)
    print("\n(relógio do Guardião adiantado 7 dias)")
    try:
        for cenario_id, texto in (
            ("contato_novo_sem_dinheiro", "Vó, a senhora viu a foto que eu mandei?"),
            ("golpe_classico", "Vó, e o Pix? Preciso hoje."),
        ):
            cenario = obter_cenario(cenario_id)
            remetente = cenario.remetente_id
            print(f"== {remetente} ({cenario_id})")
            injetar_mostrando(canal.injetar_mensagem, remetente, texto, de=cenario.remetente_nome)
            print(f"   agora: {guardiao.filtro.situacao(remetente)}")
    finally:
        RELOGIO.voltar()

### Celular simulado da Dona Maria

Mostra o que chegou ao mensageiro dela e os avisos do Guardião (marcados com `>>`).
O aviso à pessoa idosa nunca sai pelo bot no modo demonstração: aparece aqui. A célula
manda um aviso de teste só para mostrar o celular; o aviso automático (golpe confirmado ou
10 minutos sem resposta) vem numa próxima versão.

In [ ]:
from guardiao.canal.simulado import formatar_celular

if MODO != "demonstracao":
    print("aviso: o celular simulado só existe no modo demonstração; pulando")
else:
    canal.enviar_aviso(canal.id_idosa, "Teste do Guardião: este é o celular simulado da Dona Maria.")
    print(formatar_celular(canal.celular()))

## 7. Cadastro da responsável

O bot é privado: só cadastra quem toca **Iniciar** pelo link com código de uso único
gerado aqui (vale 30 minutos). Abra o link no Telegram da responsável e toque em
**Iniciar**: ela recebe "Pronto, …" pelo bot. Um Iniciar sem código, com código já usado,
vencido ou inventado recebe "Este bot é privado…" e não cadastra ninguém. Depois do
cadastro, não há link novo.

Sem `TELEGRAM_BOT_TOKEN` não há bot, e a célula só avisa: recepção e filtro continuam
funcionando.

In [ ]:
from guardiao.cadastro import JaCadastrada
from guardiao.recepcao import SemBot

if guardiao is None:
    print("aviso: o Guardião não está conectado; rode a célula da seção 5")
else:
    try:
        LINK_CADASTRO = guardiao.link_cadastro()
    except SemBot as erro:
        print(f"aviso: {erro}")
    except JaCadastrada:
        print("a responsável já está cadastrada: não há link novo")
    except Exception as erro:
        print(f"aviso: não consegui falar com o bot ({type(erro).__name__})")
    else:
        print("No Telegram da responsável, abra o link abaixo e toque em Iniciar")
        print("(uso único, vale 30 minutos):")
        print(LINK_CADASTRO)

### Estado do Guardião

Rode de novo depois de tocar Iniciar: "responsável cadastrada" passa a "sim".

In [ ]:
if guardiao is None:
    print("aviso: o Guardião não está conectado; rode a célula da seção 5")
else:
    estado = guardiao.estado()
    sim_nao = {True: "sim", False: "não"}
    print(f"conta da pessoa idosa conectada:  {sim_nao[estado['conectado']]}")
    print(f"pessoa idosa habilitada no bot:   {sim_nao[estado['idosa_habilitada']]}")
    print(f"bot do Guardião:                  {sim_nao[estado['tem_bot']]}")
    print(f"responsável cadastrada: {sim_nao[estado['responsavel_cadastrada']]}")
    print(f"contatos conhecidos: {estado['conhecidos']}; contatos novos observados: {estado['observados']}")

## 8. Tela do familiar

Sobe o servidor Flask numa thread e mostra a tela dentro do notebook com
`serve_kernel_port_as_iframe`. A tela mostra o estado do Guardião conectado acima e lê `api/estado` por caminho relativo, o que funciona
no iframe do Colab. Fora do Colab, abra o endereço impresso no navegador.

In [ ]:
from guardiao.app import definir_guardiao, iniciar_em_segundo_plano

definir_guardiao(guardiao)  # a tela mostra o estado deste Guardião

try:
    servidor = iniciar_em_segundo_plano()
except OSError as erro:
    servidor = None
    print(f"aviso: não consegui subir o Flask ({type(erro).__name__}: porta em uso?)")
if servidor is not None:
    print(f"Flask no ar na porta {servidor.port}")
    if EM_COLAB:
        from google.colab import output

        output.serve_kernel_port_as_iframe(servidor.port, path="/", height=420)
    else:
        print(f"aviso: fora do Colab não há iframe; abra http://127.0.0.1:{servidor.port}/")

## 9. Encerrar

Fecha o Guardião (para o polling do bot e desconecta o canal) e desliga a tela. Rode ao terminar a demonstração.

In [ ]:
from guardiao.app import definir_guardiao, parar_servidor

if globals().get("guardiao") is not None:
    guardiao.fechar()
elif globals().get("canal") is not None:
    canal.fechar()
canal = guardiao = None
definir_guardiao(None)
parar_servidor()
print("Guardião encerrado")